In [2]:
from osgeo import gdal
import xarray as xr
import numpy as np
import rioxarray
import os
from datetime import datetime
import requests

<h1>Download HAR v2</h1>

In [ ]:
from osgeo import gdal
import xarray as xr
import numpy as np
import rioxarray
import os
from datetime import datetime
import requests
import logging
import re
from pathlib import Path
from tqdm import tqdm
import geopandas as gpd
##########################################################################
#   ____  _____ _____ _   _ ____  
#  / ___|| ____|_   _| | | |  _ \ 
#  \___ \|  _|   | | | | | | |_) |
#   ___) | |___  | | | |_| |  __/ 
#  |____/|_____| |_|  \___/|_|    
#########################################################################
har_crs = (
    f"+proj=lcc "
    f"+lat_1={float(32.0)} "
    f"+lat_2={float(38.0)} "
    f"+lat_0={float(31.99998856)} "
    f"+lon_0={float(83.0)} "
    f"+x_0={float(0.0)} "
    f"+y_0={float(0.0)} "
    f"+a={float(6370000.0)} "
    f"+b={float(6370000.0)} "
    f"+units=m +no_defs"
)
base_url = "https://data.klima.tu-berlin.de/HAR/v2/d10km/h/2d/"
base_dir = os.path.dirname(os.getcwd()) 
out_dir_nc = os.path.join(base_dir, "Data", "Datasets", "Raw", "har_datasets", "1_nc_prcp") #original
out_dir_hr = os.path.join(base_dir, "Data", "Datasets", "Raw", "har_datasets", "2_hr_prcp") #original subdatasets
out_dir_hr_clip_reproject = os.path.join(base_dir, "Data", "Datasets", "Raw", "har_datasets", "3_daily_prcp") #agg daily nepal time
# out_dir_daily_roi = os.path.join(base_dir, "Data", "Datasets", "Raw", "har_datasets", "4_daily_prcp_roi") #final clipped wgs84
roi = gpd.read_file(r"E:\work\climate_downscaling\Data\Datasets\roi_nepal_rectangle.zip")
roi.to_crs(crs=har_crs, inplace=True)

logging .basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
#####################################################################
#   ____                      _                 _   _   _      _    ____ ____  _____ 
#  |  _ \  _____      ___ __ | | ___   __ _  __| | | \ | | ___| |_ / ___|  _ \|  ___|
#  | | | |/ _ \ \ /\ / / '_ \| |/ _ \ / _` |/ _` | |  \| |/ _ \ __| |   | | | | |_   
#  | |_| | (_) \ V  V /| | | | | (_) | (_| | (_| | | |\  |  __/ |_| |___| |_| |  _|  
#  |____/ \___/ \_/\_/ |_| |_|_|\___/ \__,_|\__,_| |_| \_|\___|\__|\____|____/|_|                                                                         
#####################################################################
def get_file_links(base_url: str, keyword: str = "prcp"):
    logging.info(f"Listing files from {base_url} (keyword={keyword})")
    r = requests.get(base_url)
    r.raise_for_status()
    nc_files = re.findall(r'href="(.*?\.nc)"', r.text)
    filtered = [x for x in nc_files if keyword in x]
    logging.info(f"Found {len(filtered)} matching files")
    return filtered

def download_nc(filename, base_url, dest):
    os.makedirs(dest, exist_ok=True)
    url = base_url + filename
    dest_path = dest / filename
    if dest_path.exists():
        logging.info(f"Already downloaded: {filename}")
        return dest_path
    
    logging.info(f"Downloading {url}")

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        
        total_size = int(r.headers.get('content-length', 0))
        with tqdm(total=total_size, unit='iB', unit_scale=True, desc=filename) as bar:
            with open(dest_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=10 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
                        
    logging.info(f"Saved to {dest_path}")
    return dest_path
####################################################################
#   _____      _                  _     ____        _       ____        _        
#  | ____|_  _| |_ _ __ __ _  ___| |_  / ___| _   _| |__   |  _ \  __ _| |_ __ _ 
#  |  _| \ \/ / __| '__/ _` |/ __| __| \___ \| | | | '_ \  | | | |/ _` | __/ _` |
#  | |___ >  <| |_| | | (_| | (__| |_   ___) | |_| | |_) | | |_| | (_| | || (_| |
#  |_____/_/\_\\__|_|  \__,_|\___|\__| |____/ \__,_|_.__/  |____/ \__,_|\__\__,_|
###################################################################
def extract_nc_subdatasets(
        in_nc_dir, 
        out_tif_dir,
        x_dim, 
        y_dim,
        crs,
        variable_to_extract,
        dimension,
        time_offset
        ):
    os.makedirs(out_tif_dir, exist_ok=True)
    nc_file_list = [nc for nc in os.listdir(in_nc_dir) if nc.endswith(".nc")]
    for nc in nc_file_list:
        nc_full_path = os.path.join(in_nc_dir, nc)
        outpath_tif = "Unknown File"
        try:
            nc_ds = xr.open_dataset(nc_full_path)

            
            for index, time in enumerate(nc_ds[dimension].values):
                original_time = np.datetime64(time)
                time_str = str(original_time)[:13].replace("T", "-")
                if time_offset is not None:
                    time_offset_td = np.timedelta64(time_offset, 'h')
                    new_time = original_time + time_offset_td
                    time_str = str(new_time)[:13].replace("T", "-")

                outpath_tif = os.path.join(out_tif_dir, f"{variable_to_extract}_{time_str}.tif")
                arr_slice = nc_ds[variable_to_extract].isel(time=index)
                arr_slice.rio.set_spatial_dims(x_dim=x_dim, y_dim = y_dim, inplace=True)
                arr_slice.rio.write_crs(crs, inplace=True)
                arr_slice.rio.to_raster(outpath_tif)
                logging.info(f"Saved {outpath_tif}")
        except Exception as e:
            nc_ds = None
            logging.error(f"Failed to save {outpath_tif}: {e}")
########################################################################
#    ____ _ _                         _   ____                      _           _   
#   / ___| (_)_ __     __ _ _ __   __| | |  _ \ ___ _ __  _ __ ___ (_) ___  ___| |_ 
#  | |   | | | '_ \   / _` | '_ \ / _` | | |_) / _ \ '_ \| '__/ _ \| |/ _ \/ __| __|
#  | |___| | | |_) | | (_| | | | | (_| | |  _ <  __/ |_) | | | (_) | |  __/ (__| |_ 
#   \____|_|_| .__/   \__,_|_| |_|\__,_| |_| \_\___| .__/|_|  \___// |\___|\___|\__|
#            |_|                                   |_|           |__/               
#########################################################################
def clip_reproject_tif(in_tif_dir, out_tif_dir, clip_geom, out_crs):
    os.makedirs(out_tif_dir, exist_ok=True)
    tif_file_list = [tif for tif in os.listdir(in_tif_dir) if tif.endswith(".tif")]
    ref_tif = xr.open_dataset(os.path.join(in_tif_dir, tif_file_list[0]))
    if clip_geom.crs != ref_tif.rio.crs:
            clip_geom = clip_geom.to_crs(crs=ref_tif.crs)
    ref_tif = ref_tif.rio.clip(clip_geom.geometry.values, clip_geom.crs)
    ref_tif = ref_tif.rio.reproject(out_crs)
    logging.info(f"Ref template created from {tif_file_list[0]} for matching")

    for tif in tif_file_list:
        tif_full_path = os.path.join(in_tif_dir, tif)
        out_tif_path = os.path.join(out_tif_dir, tif)
        try:
            ds = xr.open_dataset(tif_full_path)
            clipped_tif = ds.rio.clip(clip_geom.geometry.values, clip_geom.crs)
            reprojected_tif = clipped_tif.rio.reproject_match(ref_tif)
            reprojected_tif.rio.to_raster(out_tif_path)
            logging.info(f"Saved {out_tif_path}")

        except Exception as e:
            logging.error(f"Could no reproject/clip {tif}: {e}")

In [74]:
ds = xr.open_dataset(r"E:\work\climate_downscaling\Data\Datasets\Raw\har_datasets\2_hr_prcp\prcp_2018-01-01-07.tif")


In [79]:
roi.geometry.values

<GeometryArray>
[<POLYGON ((-336798.727 -643668.418, -320281.643 -150430.659, -316980.584 -15...>]
Length: 1, dtype: geometry

In [71]:
ds.rio.crs

CRS.from_wkt('PROJCS["unknown",GEOGCS["unknown",DATUM["unknown",SPHEROID["unknown",6370000,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",31.99998856],PARAMETER["central_meridian",83],PARAMETER["standard_parallel_1",32],PARAMETER["standard_parallel_2",38],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]')

In [67]:
gdf.crs

<Projected CRS: +proj=lcc +lat_1=32.0 +lat_2=38.0 +lat_0=31.999988 ...>
Name: unknown
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- undefined
Coordinate Operation:
- name: unknown
- method: Lambert Conic Conformal (2SP)
Datum: unknown
- Ellipsoid: unknown
- Prime Meridian: Greenwich

In [59]:
extract_nc_subdatasets(
    in_nc_dir = r"C:\Users\Suraj2\Downloads\Documents\page_6_10", 
    out_tif_dir= out_dir_hr,
    x_dim = "west_east", 
    y_dim = "south_north",
    spatial_reference = har_crs,
    variable_to_extract = "prcp",
    dimension = "time",
    time_offset = 6
)

2025-11-20 14:57:44,535 - INFO - Saved e:\work\climate_downscaling\Data\Datasets\Raw\har_datasets\2_hr_prcp\prcp_2018-01-01-07.tif
2025-11-20 14:57:44,557 - INFO - Saved e:\work\climate_downscaling\Data\Datasets\Raw\har_datasets\2_hr_prcp\prcp_2018-01-01-08.tif
2025-11-20 14:57:44,578 - INFO - Saved e:\work\climate_downscaling\Data\Datasets\Raw\har_datasets\2_hr_prcp\prcp_2018-01-01-09.tif
2025-11-20 14:57:44,601 - INFO - Saved e:\work\climate_downscaling\Data\Datasets\Raw\har_datasets\2_hr_prcp\prcp_2018-01-01-10.tif
2025-11-20 14:57:44,623 - INFO - Saved e:\work\climate_downscaling\Data\Datasets\Raw\har_datasets\2_hr_prcp\prcp_2018-01-01-11.tif
2025-11-20 14:57:44,645 - INFO - Saved e:\work\climate_downscaling\Data\Datasets\Raw\har_datasets\2_hr_prcp\prcp_2018-01-01-12.tif
2025-11-20 14:57:44,666 - INFO - Saved e:\work\climate_downscaling\Data\Datasets\Raw\har_datasets\2_hr_prcp\prcp_2018-01-01-13.tif
2025-11-20 14:57:44,688 - INFO - Saved e:\work\climate_downscaling\Data\Datasets\Ra

KeyboardInterrupt: 

In [12]:
netcdf_file_path = r"C:\Users\Suraj2\Downloads\Documents\page_6_10\HARv2_d10km_y_2d_prcp_nc_2020.nc"
variable_name = "prcp"

In [13]:
ds =xr.open_dataset(netcdf_file_path)

In [14]:
ds = ds.rio.set_spatial_dims(x_dim="west_east", y_dim="south_north", inplace=True)

In [26]:
for i, t in enumerate(ds.time.values):
    print(str(t)[:13].replace("T", "-"))
    break

2020-01-01-00


In [ ]:
raster_crs = (
    f"+proj=lcc "
    f"+lat_1={float(ds.PROJ_STANDARD_PAR1.strip())} "
    f"+lat_2={float(ds.PROJ_STANDARD_PAR2.strip())} "
    f"+lat_0={float(ds.PROJ_CENTRAL_LAT.strip())} "
    f"+lon_0={float(ds.PROJ_CENTRAL_LON.strip())} "
    f"+x_0={float(ds.PROJ_FALSE_EASTING.strip())} "
    f"+y_0={float(ds.PROJ_FALSE_NORTHING.strip())} "
    f"+a={float(ds.PROJ_SEMIMAJOR_AXIS.strip())} "
    f"+b={float(ds.PROJ_SEMIMINOR_AXIS.strip())} "
    f"+units=m +no_defs"
)

In [47]:
ds = ds.rio.write_crs(raster_crs, inplace=True) # Example for WGS84 (lat/lon)


In [49]:
ds_reprojected = ds.rio.reproject("EPSG:4326")

In [51]:
ds_reprojected.rio.crs

CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]')

In [52]:
output_dir = r"E:\work\climate_downscaling\Data\Raw\har_datasets\delete"

In [ ]:
nepal_time_offset = np.timedelta64(6, 'h')

for i, t in enumerate(ds.time.values):
    original_time = np.datetime64(t)
    nepal_time = original_time + nepal_time_offset
    t_str = str(nepal_time)[:13].replace("-", "")
    out_path = os.path.join(output_dir, f"{variable_name}_{t_str}.tif")
    # Select the variable at this time step
    ds_reprojected[variable_name].isel(time=i).rio.to_raster(out_path)
    print(f"Saved {out_path}")

Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T01.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T02.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T03.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T04.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T05.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T06.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T07.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T08.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T09.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T10.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T11.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101

In [88]:
ds_reprojected["time"].isel(time = 0) + nepal_time

<xarray.DataArray 'time' ()> Size: 8B
array('2020-01-01T07:00:00.000000000', dtype='datetime64[ns]')
Coordinates:
    time         datetime64[ns] 8B 2020-01-01T01:00:00
    spatial_ref  int64 8B 0
Attributes:
    long_name:  Time

In [78]:
dir_path = r"E:\work\climate_downscaling\Data\Raw\har_datasets\delete"
file_list = os.listdir(dir_path)
tif_list = [file for file in file_list if file.endswith(".tif")]

In [79]:
import numpy as np
import rasterio

# Read first file and get nodata value
with rasterio.open(os.path.join(dir_path, tif_list[0])) as s0:
    profile = s0.profile
    nodata = profile.get('nodata', None)
    # print(nodata)
    first_arr = s0.read(1).astype(np.float32)
    
    # Create masked array if nodata exists
    if nodata is not None:
        sum_arr = np.ma.masked_equal(first_arr, nodata)
    else:
        sum_arr = first_arr

# Sum remaining files
for raster in tif_list[1:]:
    full_raster_path = os.path.join(dir_path, raster)
    print(full_raster_path)
    with rasterio.open(full_raster_path) as s:
        arr = s.read(1).astype(np.float32)
        
        # Mask nodata values
        if nodata is not None:
            arr = np.ma.masked_equal(arr, nodata)
        
        sum_arr += arr

# Convert back to regular array with nodata filled
if nodata is not None:
    sum_arr = sum_arr.filled(nodata)

# Write output
with rasterio.open(r"E:\work\climate_downscaling\Data\Raw\har_datasets\sum2020_hr_xarray.tif", 
                   "w", **profile) as dst:
    dst.write(sum_arr.astype(np.float32), 1)

nan
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T02.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T03.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T04.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T05.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T06.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T07.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T08.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T09.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T10.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T11.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T12.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T13.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prc

In [3]:
from colorama import Fore as f
from colorama import Back as b
from colorama import Style as s

In [6]:
print(f.BLUE+ "Blue")
print(f.GREEN+ "GREEN")
print(f.RED+ "RED")
print(s.RESET_ALL)
print("None")

Blue
GREEN
RED

None


In [4]:
from osgeo import gdal
import xarray as xr
import numpy as np
import rioxarray
import os
from datetime import datetime
import requests
import logging
import re
from pathlib import Path
from tqdm import tqdm
import geopandas as gpd
from colorama import Fore, Style

In [8]:
def calculate_daily_total_precipitation(in_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    
    # List all TIF files
    tif_files = [f for f in os.listdir(in_dir) if f.endswith('.tif')]
    if not tif_files:
        logging.warning(Fore.YELLOW + f"No TIF files found in {in_dir}")
        return

    # Group files by date (YYYY-MM-DD)
    files_by_date = {}
    for tif in tif_files:
        # Expected format: prcp_YYYY-MM-DD-HH.tif
        # Extract YYYY-MM-DD
        try:
            # Assuming format ends with YYYY-MM-DD-HH.tif
            # Example: prcp_2020-01-01-06.tif
            date_str = tif.split('_')[-1][:10] # Get '2020-01-01'
            if date_str not in files_by_date:
                files_by_date[date_str] = []
            files_by_date[date_str].append(tif)
        except Exception as e:
            logging.warning(Fore.YELLOW + f"Skipping file with unexpected name format {tif}: {e}")
    print(files_by_date)
    # Process each day
    # for date_str, files in sorted(files_by_date.items()):
    #     out_path = os.path.join(out_dir, f"prcp_{date_str}.tif")
        
    #     if os.path.exists(out_path):
    #         logging.info(f"Skipping {date_str} (already exists)")
    #         continue
            
    #     logging.info(Fore.BLUE + f"Calculating daily total for {date_str} from {len(files)} files...")
        
    #     try:
    #         daily_total = None
            
    #         for tif in files:
    #             tif_path = os.path.join(in_dir, tif)
    #             # Open raster
    #             da = rioxarray.open_rasterio(tif_path)
                
    #             if daily_total is None:
    #                 daily_total = da
    #             else:
    #                 daily_total = daily_total + da
            
    #         if daily_total is not None:
    #             daily_total.rio.to_raster(out_path)
    #             logging.info(Fore.GREEN + f"Saved {out_path}")
                
    #     except Exception as e:
    #         logging.error(Fore.RED + f"Failed to calculate daily total for {date_str}: {e}")

In [9]:
calculate_daily_total_precipitation(r"E:\work\Data\Datasets\Raw\har_datasets\2_hr_prcp", r"E:\work\Data\Datasets\Raw\har_datasets\4_daily_prcp")

{'2020-01-01': ['prcp_2020-01-01-07.tif', 'prcp_2020-01-01-08.tif', 'prcp_2020-01-01-09.tif', 'prcp_2020-01-01-10.tif', 'prcp_2020-01-01-11.tif', 'prcp_2020-01-01-12.tif', 'prcp_2020-01-01-13.tif', 'prcp_2020-01-01-14.tif', 'prcp_2020-01-01-15.tif', 'prcp_2020-01-01-16.tif', 'prcp_2020-01-01-17.tif', 'prcp_2020-01-01-18.tif', 'prcp_2020-01-01-19.tif', 'prcp_2020-01-01-20.tif', 'prcp_2020-01-01-21.tif', 'prcp_2020-01-01-22.tif', 'prcp_2020-01-01-23.tif'], '2020-01-02': ['prcp_2020-01-02-00.tif', 'prcp_2020-01-02-01.tif', 'prcp_2020-01-02-02.tif', 'prcp_2020-01-02-03.tif', 'prcp_2020-01-02-04.tif', 'prcp_2020-01-02-05.tif', 'prcp_2020-01-02-06.tif', 'prcp_2020-01-02-07.tif', 'prcp_2020-01-02-08.tif', 'prcp_2020-01-02-09.tif', 'prcp_2020-01-02-10.tif', 'prcp_2020-01-02-11.tif', 'prcp_2020-01-02-12.tif', 'prcp_2020-01-02-13.tif', 'prcp_2020-01-02-14.tif', 'prcp_2020-01-02-15.tif', 'prcp_2020-01-02-16.tif', 'prcp_2020-01-02-17.tif', 'prcp_2020-01-02-18.tif', 'prcp_2020-01-02-19.tif', 'prcp_

In [10]:
tif_files = [f for f in os.listdir(r"E:\work\Data\Datasets\Raw\har_datasets\2_hr_prcp") if f.endswith('.tif')]

# Group files by date (YYYY-MM-DD)
files_by_date = {}
for tif in tif_files:
    # Expected format: prcp_YYYY-MM-DD-HH.tif
    # Extract YYYY-MM-DD
    try:
        # Assuming format ends with YYYY-MM-DD-HH.tif
        # Example: prcp_2020-01-01-06.tif
        date_str = tif.split('_')[-1][:10] # Get '2020-01-01'
        if date_str not in files_by_date:
            files_by_date[date_str] = []
        files_by_date[date_str].append(tif)
    except Exception as e:
        logging.warning(Fore.YELLOW + f"Skipping file with unexpected name format {tif}: {e}")
print(files_by_date)

{'2020-01-01': ['prcp_2020-01-01-07.tif', 'prcp_2020-01-01-08.tif', 'prcp_2020-01-01-09.tif', 'prcp_2020-01-01-10.tif', 'prcp_2020-01-01-11.tif', 'prcp_2020-01-01-12.tif', 'prcp_2020-01-01-13.tif', 'prcp_2020-01-01-14.tif', 'prcp_2020-01-01-15.tif', 'prcp_2020-01-01-16.tif', 'prcp_2020-01-01-17.tif', 'prcp_2020-01-01-18.tif', 'prcp_2020-01-01-19.tif', 'prcp_2020-01-01-20.tif', 'prcp_2020-01-01-21.tif', 'prcp_2020-01-01-22.tif', 'prcp_2020-01-01-23.tif'], '2020-01-02': ['prcp_2020-01-02-00.tif', 'prcp_2020-01-02-01.tif', 'prcp_2020-01-02-02.tif', 'prcp_2020-01-02-03.tif', 'prcp_2020-01-02-04.tif', 'prcp_2020-01-02-05.tif', 'prcp_2020-01-02-06.tif', 'prcp_2020-01-02-07.tif', 'prcp_2020-01-02-08.tif', 'prcp_2020-01-02-09.tif', 'prcp_2020-01-02-10.tif', 'prcp_2020-01-02-11.tif', 'prcp_2020-01-02-12.tif', 'prcp_2020-01-02-13.tif', 'prcp_2020-01-02-14.tif', 'prcp_2020-01-02-15.tif', 'prcp_2020-01-02-16.tif', 'prcp_2020-01-02-17.tif', 'prcp_2020-01-02-18.tif', 'prcp_2020-01-02-19.tif', 'prcp_

In [15]:
error = []
for date_str, files in sorted(files_by_date.items()):
    print(date_str)
    print(len(files))
    for file in files:
        print(file)
    if len(files) != 24:
        error.append(date_str)

2020-01-01
17
prcp_2020-01-01-07.tif
prcp_2020-01-01-08.tif
prcp_2020-01-01-09.tif
prcp_2020-01-01-10.tif
prcp_2020-01-01-11.tif
prcp_2020-01-01-12.tif
prcp_2020-01-01-13.tif
prcp_2020-01-01-14.tif
prcp_2020-01-01-15.tif
prcp_2020-01-01-16.tif
prcp_2020-01-01-17.tif
prcp_2020-01-01-18.tif
prcp_2020-01-01-19.tif
prcp_2020-01-01-20.tif
prcp_2020-01-01-21.tif
prcp_2020-01-01-22.tif
prcp_2020-01-01-23.tif
2020-01-02
24
prcp_2020-01-02-00.tif
prcp_2020-01-02-01.tif
prcp_2020-01-02-02.tif
prcp_2020-01-02-03.tif
prcp_2020-01-02-04.tif
prcp_2020-01-02-05.tif
prcp_2020-01-02-06.tif
prcp_2020-01-02-07.tif
prcp_2020-01-02-08.tif
prcp_2020-01-02-09.tif
prcp_2020-01-02-10.tif
prcp_2020-01-02-11.tif
prcp_2020-01-02-12.tif
prcp_2020-01-02-13.tif
prcp_2020-01-02-14.tif
prcp_2020-01-02-15.tif
prcp_2020-01-02-16.tif
prcp_2020-01-02-17.tif
prcp_2020-01-02-18.tif
prcp_2020-01-02-19.tif
prcp_2020-01-02-20.tif
prcp_2020-01-02-21.tif
prcp_2020-01-02-22.tif
prcp_2020-01-02-23.tif
2020-01-03
24
prcp_2020-01-03

In [2]:
import os

os.path.dirname(os.getcwd()) 

'e:\\work\\climate_downscaling'